In [1]:
from ingest import load_faq_data
documents = load_faq_data()

In [2]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

153

In [3]:
documents = documents_llm

In [4]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [6]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [12]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv(override=True)
openai_client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [13]:
import json

user_prompt = json.dumps(doc)

In [14]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [15]:
response = openai_client.responses.parse(
    model="gpt-6-luna",
    input=messages,
    text_format=Questions
)

In [16]:
result = response.output_parsed

print(result)

questions=['I found this course late—can I still sign up?', 'Is it too late to join the course now?', 'Can I start the course even if it’s already underway?', 'Can I still earn a certificate if I join late?', 'How late can I submit my project and still get a certificate?']


In [22]:
from evaluation_utils import llm_structured

In [23]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — can I still sign up and take part, or is it too late now?', 'Is it still possible to join the course after it has already started?', 'If I start the course late, will I still be able to get a certificate somehow?', 'Do I need to finish everything right away, or can I join now and work on the project later?', 'What’s the deadline for the project if I want to be eligible for a certificate?']


In [24]:
usage.input_tokens, usage.output_tokens

(207, 108)

In [25]:
from evaluation_utils import calc_price

In [26]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.000486, 'total_cost': 0.00064125}

In [27]:
from evaluation_utils import llm_structured_retry

In [28]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [29]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [35]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [36]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/153 [00:00<?, ?it/s]

In [37]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

765

In [38]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.12256424999999997

In [39]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.12256424999999997

In [40]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [43]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)

In [44]:
import pandas as pd

df_ground_truth = pd.read_csv("data/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [45]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [46]:
def text_search(query):
    boost_dict = {"question": 3.0, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict
    )

In [47]:
q = ground_truth[0]
q

{'question': 'I found this course late — can I still sign up and follow along?',
 'document': '74eb249bbf'}

In [48]:
doc_id = q["document"]
results = text_search(query=q["question"])

In [49]:
for d in results:
    print(f'{d["id"]} == {doc_id}: {d["id"] == doc_id}')

04919992b3 == 74eb249bbf: False
74eb249bbf == 74eb249bbf: True
69d122f12e == 74eb249bbf: False
a9353fadfe == 74eb249bbf: False
85384a18e5 == 74eb249bbf: False


In [50]:
relevance = []

for d in results:
    relevance.append(int(d["id"] == doc_id))

relevance

[0, 1, 0, 0, 0]

In [51]:
def compute_relevance_text(q):
    doc_id = q["document"]
    results = text_search(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [52]:
q = ground_truth[0]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

I found this course late — can I still sign up and follow along?


[0, 1, 0, 0, 0]

In [53]:
q = ground_truth[11]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where should I look for the live stream URL before an office hours session starts?


[1, 0, 0, 0, 0]

In [54]:
q = ground_truth[50]
print(q["question"])
compute_relevance_text(q)
# [1, 0, 0, 0, 0]

Where do I check the LLM Zoomcamp syllabus and what the current cohort is doing?


[1, 0, 0, 0, 0]

In [55]:
from tqdm.auto import tqdm

def compute_relevance_total_text(ground_truth):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance_text(q)
        relevance_total.append(relevance)

    return relevance_total

In [56]:
ground_truth_sample = ground_truth[:15]
relevance_total_text = compute_relevance_total_text(ground_truth_sample)

  0%|          | 0/15 [00:00<?, ?it/s]

In [57]:
relevance_total_text

[[0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [58]:
def compute_relevance(q, search_function):
    doc_id = q["document"]
    results = search_function(query=q["question"])

    relevance = []
    for d in results:
        relevance.append(int(d["id"] == doc_id))

    return relevance

In [59]:
def compute_relevance_total(ground_truth, search_function):
    relevance_total = []

    for q in tqdm(ground_truth):
        relevance = compute_relevance(q, search_function)
        relevance_total.append(relevance)

    return relevance_total

In [60]:
relevance_total = compute_relevance_total(ground_truth_sample, text_search)
relevance_total

  0%|          | 0/15 [00:00<?, ?it/s]

[[0, 1, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 1, 0, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 1, 0],
 [0, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0],
 [1, 0, 0, 0, 0]]

In [61]:
relevance_total = compute_relevance_total(ground_truth, text_search)

  0%|          | 0/765 [00:00<?, ?it/s]

In [62]:
example = [
    [1, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 0, 0, 0],
    [0, 1, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [0, 0, 1, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
    [1, 0, 0, 0, 0],
]

In [63]:
cnt = 0

for line in example:
    if 1 in line:
        cnt = cnt + 1

cnt

14

In [64]:
cnt / len(example)
# 0.933

0.9333333333333333

In [65]:
def hit_rate(relevance):
    cnt = 0

    for line in relevance:
        if 1 in line:
            cnt = cnt + 1

    return cnt / len(relevance)

In [66]:
hit_rate(example)
# 0.933

0.9333333333333333

In [67]:
total_score = 0.0

for line in example:
    for rank in range(len(line)):
        if line[rank] == 1:
            total_score = total_score + 1 / (rank + 1)
            break

total_score

12.333333333333332

In [68]:
total_score / len(example)
# 0.822

0.8222222222222222

In [69]:
def mrr(relevance):
    total_score = 0.0

    for line in relevance:
        for rank in range(len(line)):
            if line[rank] == 1:
                total_score = total_score + 1 / (rank + 1)
                break

    return total_score / len(relevance)

In [70]:
mrr(example)
# 0.822

0.8222222222222222

In [71]:
def evaluate(ground_truth, search_function):
    relevance_total = compute_relevance_total(ground_truth, search_function)

    return {
        "hit_rate": hit_rate(relevance_total),
        "mrr": mrr(relevance_total),
    }

In [72]:
evaluate(
    ground_truth,
    text_search
)

  0%|          | 0/765 [00:00<?, ?it/s]

{'hit_rate': 0.8405228758169935, 'mrr': 0.7164488017429191}

In [73]:
def search_boost(query, question_boost):
    boost_dict = {"question": question_boost, "section": 0.5}

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [74]:
for boost in [0.5, 1.0, 3.0, 5.0, 10.0]:
    result = evaluate(
        ground_truth,
        lambda query, boost=boost: search_boost(query, boost)
    )
    print(f"boost={boost}: {result}")

  0%|          | 0/765 [00:00<?, ?it/s]

boost=0.5: {'hit_rate': 0.8941176470588236, 'mrr': 0.7979956427015251}


  0%|          | 0/765 [00:00<?, ?it/s]

boost=1.0: {'hit_rate': 0.8915032679738563, 'mrr': 0.7826797385620912}


  0%|          | 0/765 [00:00<?, ?it/s]

boost=3.0: {'hit_rate': 0.8405228758169935, 'mrr': 0.7164488017429191}


  0%|          | 0/765 [00:00<?, ?it/s]

boost=5.0: {'hit_rate': 0.8130718954248366, 'mrr': 0.6776906318082784}


  0%|          | 0/765 [00:00<?, ?it/s]

boost=10.0: {'hit_rate': 0.7738562091503268, 'mrr': 0.6452069716775595}


In [75]:
def search_boosts(query, question_boost, answer_boost, section_boost):
    boost_dict = {
        "question": question_boost,
        "section": section_boost,
        "answer": answer_boost,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )

In [76]:
results = []

for question_boost in [1.0, 2.0, 5.0]:
    for answer_boost in [1.0, 2.0, 4.0, 10.0]:
        for section_boost in [0.1, 0.2, 0.5]:
            print(
                f"Evaluating question_boost={question_boost},"
                f" answer_boost={answer_boost},"
                f" section_boost={section_boost}..."
            )
            result = evaluate(
                ground_truth,
                lambda query, question_boost=question_boost, answer_boost=answer_boost, section_boost=section_boost: search_boosts(
                    query,
                    question_boost,
                    answer_boost,
                    section_boost
                )
            )

            results.append({
                "question": question_boost,
                "answer": answer_boost,
                "section": section_boost,
                "hit_rate": result["hit_rate"],
                "mrr": result["mrr"],
            })

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.1...


  0%|          | 0/765 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.2...


  0%|          | 0/765 [00:00<?, ?it/s]

Evaluating question_boost=1.0, answer_boost=1.0, section_boost=0.5...


...

  0%|          | 0/765 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.2...


  0%|          | 0/765 [00:00<?, ?it/s]

Evaluating question_boost=5.0, answer_boost=10.0, section_boost=0.5...


  0%|          | 0/765 [00:00<?, ?it/s]

In [77]:
df_results = pd.DataFrame(results)
df_results.sort_values("mrr", ascending=False).head(10)

,question,answer,section,hit_rate,mrr
3,1.0,2.0,0.1,0.967320,0.875229
19,2.0,4.0,0.2,0.967320,0.875229
35,5.0,10.0,0.5,0.967320,0.875229
18,2.0,4.0,0.1,0.966013,0.874641
4,1.0,2.0,0.2,0.967320,0.874466
34,5.0,10.0,0.2,0.966013,0.874205
33,5.0,10.0,0.1,0.963399,0.872266
20,2.0,4.0,0.5,0.967320,0.871002
7,1.0,4.0,0.2,0.964706,0.869041
6,1.0,4.0,0.1,0.963399,0.868126


In [78]:
def text_search(query):
    boost_dict = {
        "question": 1.0,
        "answer": 2.0,
        "section": 0.1,
    }

    return index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
    )